# Paredes por estilo

> **Qué compara.** En FloorPlanCAD la mayoría de las paredes tiene líneas adentro: el eje, capas o un rayado. En BLD-AR casi todas son una doble línea vacía. Si el modelo aprendió a reconocer las paredes por esas líneas, va a encontrar peor las vacías, y eso explicaría parte de su caída en BLD-AR.
>
> **Cómo lo mide.**
> 1. Clasifica cada pared etiquetada según lo que tiene adentro: vacía, con líneas, relleno gris o relleno negro.
> 2. Pasa el modelo por cada plano, igual que en la evaluación en BLD-AR, y mide qué parte de cada pared marca. También calcula el AP50 de seg de pared de cada grupo.
> 3. Compara las paredes vacías con las que tienen líneas dentro del test de FloorPlanCAD: ahí solo cambia el estilo, no el conjunto. BLD-AR se muestra como referencia, porque casi no tiene paredes con líneas.
>
> **Cómo se decide.** En el test de FloorPlanCAD se calculan dos promedios: A, qué parte de cada pared vacía marca el modelo, y B, qué parte de cada pared con líneas adentro. Después se divide A / B. Por ejemplo, si a las vacías les marca el 60 % y a las con líneas el 90 %, A / B = 60 / 90 = 67 %.
>
> | A / B | Qué quiere decir | Hipótesis |
> |---|---|---|
> | Menos del 80 % | Encuentra claramente peor las paredes vacías | Confirmada |
> | Entre 80 % y 95 % | Las encuentra algo peor | Confirmada en parte |
> | 95 % o más | Las encuentra casi igual: las líneas no importan | Descartada |

In [ ]:
!nvidia-smi
!for pid in $(nvidia-smi --query-compute-apps=pid --format=csv,noheader); do ps -o user=,pid=,etime=,cmd= -p $pid; done

In [ ]:
%pip install ultralytics==8.4.173 sahi==0.12.8 pycocotools==2.0.11

## Imports

In [ ]:
import json
from pathlib import Path

import numpy as np
import shapely
import yaml
from IPython.display import Markdown, display
from PIL import Image, ImageDraw
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from sahi.utils.cv import read_image_as_pil
from shapely.geometry import Polygon

## Rutas y parámetros

In [ ]:
REPO = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "datasets").is_dir()), Path.cwd())
BLDAR = REPO / "datasets/BLD-AR"
FLOORPLANCAD = REPO / "datasets/floor-plan-cad-yolo"
WEIGHTS = REPO / "weights/00-YOLO26m-SEG-1024/best.pt"

TILE = 1024
OVERLAP = 0.2
CONF = 0.05        # como en la evaluación en BLD-AR, para el AP50 de seg
PIXEL_CONF = 0.25  # desde esta confianza una predicción marca pared, para la pared encontrada
DEVICE = "cuda:0"

# Clases de BLD-AR que se evalúan (docs/MAPEO_CLASES.md); las demás son zonas a ignorar
MAPPED = {"wall", "door", "window", "sliding_door", "stair", "elevator", "railing"}
STYLES = ["Vacía", "Con líneas adentro", "Relleno gris", "Relleno negro"]

assert WEIGHTS.exists(), f"No existen los pesos: {WEIGHTS}"

## Modelo

In [ ]:
model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics", model_path=str(WEIGHTS),
    confidence_threshold=CONF, device=DEVICE, image_size=TILE,
)

## Formas y máscaras

In [ ]:
def shape_points(shape):
    """Puntos de una forma LabelMe; los círculos se aproximan con 32 lados."""
    points = np.array(shape["points"], dtype=float)
    if shape["shape_type"] == "circle":
        center, edge = points
        angles = np.linspace(0, 2 * np.pi, 32, endpoint=False)
        points = center + np.linalg.norm(edge - center) * np.stack([np.cos(angles), np.sin(angles)], axis=1)
    return points


def to_geometry(points):
    """Polígono válido a partir de sus puntos; puede quedar vacío o partido en varios."""
    if len(points) < 3:
        return Polygon()
    parts = [g for g in shapely.get_parts(shapely.make_valid(Polygon(points))) if g.geom_type == "Polygon"]
    return shapely.unary_union(parts) if parts else Polygon()


def polygons(geometry):
    return [g for g in shapely.get_parts(geometry) if g.geom_type == "Polygon" and not g.is_empty]


def region_mask(geometry, box):
    """Máscara de una geometría dentro del recuadro box = (x0, y0, x1, y1) del plano."""
    canvas = Image.new("1", (box[2] - box[0], box[3] - box[1]))
    draw = ImageDraw.Draw(canvas)
    for part in polygons(geometry):
        draw.polygon([(x - box[0], y - box[1]) for x, y in part.exterior.coords], fill=1)
    return np.array(canvas)


def clipped_box(geometry, size):
    x0, y0, x1, y1 = geometry.bounds
    return max(int(x0), 0), max(int(y0), 0), min(int(x1) + 1, size[0]), min(int(y1) + 1, size[1])

## Estilo de cada pared

> Se mira el interior de la pared, sin los bordes: relleno negro si más del 60 % es oscuro, relleno gris si más de la mitad es gris, con líneas adentro si más del 5 % es oscuro; si no, vacía. Las paredes demasiado finas para tener interior quedan sin clasificar.

In [ ]:
def wall_style(picture, wall):
    """Estilo de una pared según su interior; None si no tiene interior."""
    thickness = 2 * wall.area / wall.length
    inner = wall.buffer(-max(2.0, 0.25 * thickness))
    if inner.is_empty:
        return None
    box = clipped_box(inner, picture.size)
    values = np.asarray(picture.crop(box))[region_mask(inner, box)]
    if values.size == 0:
        return None
    dark, gray = (values < 128).mean(), ((values >= 128) & (values < 230)).mean()
    if dark > 0.6:
        return "Relleno negro"
    if gray > 0.5:
        return "Relleno gris"
    return "Con líneas adentro" if dark > 0.05 else "Vacía"

## Pared encontrada en cada plano

In [ ]:
def predicted_walls(image_path):
    """Predicciones de wall por tiles, con los mismos parámetros que la evaluación en BLD-AR."""
    result = get_sliced_prediction(
        str(image_path), model,
        slice_height=TILE, slice_width=TILE,
        overlap_height_ratio=OVERLAP, overlap_width_ratio=OVERLAP,
        perform_standard_pred=False, verbose=0,
        # Con confianza < 0.1 SAHI cambia a NMS y no une los muros cortados entre tiles
        force_postprocess_type=True,
    )
    return [p for p in result.to_coco_predictions() if p["category_name"] == "wall"]


def measure_plan(result, image_path, size, walls, ignored=()):
    """Estilo y pared encontrada de cada pared del plano; guarda etiquetas y predicciones para el AP50 de seg."""
    image_id = len(result["images"]) + 1
    result["images"].append({"id": image_id, "width": size[0], "height": size[1]})
    picture = read_image_as_pil(str(image_path)).convert("L")
    predictions = predicted_walls(image_path)
    marked = Image.new("1", size)
    draw = ImageDraw.Draw(marked)
    for prediction in predictions:
        if prediction["score"] >= PIXEL_CONF:
            for polygon in prediction["segmentation"]:
                draw.polygon(polygon, fill=1)
    marked = np.array(marked)
    for wall in walls:
        box = clipped_box(wall, size)
        mask = region_mask(wall, box)
        found = (marked[box[1]:box[3], box[0]:box[2]] & mask).sum() / mask.sum() if mask.any() else np.nan
        result["walls"].append((image_id, wall, wall_style(picture, wall), found))
    result["ignored"] += [(image_id, zone) for zone in ignored]
    result["predictions"] += [{**p, "image_id": image_id, "category_id": 1} for p in predictions]


results = {name: {"images": [], "walls": [], "ignored": [], "predictions": []} for name in ("Test de FloorPlanCAD", "BLD-AR")}

## FloorPlanCAD (test)

In [ ]:
names = yaml.safe_load((FLOORPLANCAD / "floor-plan-cad-yolo.yaml").read_text())["names"]
wall_id = next(i for i, name in names.items() if name == "wall")

for image_path in sorted((FLOORPLANCAD / "images/test").iterdir()):
    size = Image.open(image_path).size
    rows = [line.split() for line in (FLOORPLANCAD / "labels/test" / f"{image_path.stem}.txt").read_text().splitlines()]
    # Etiqueta YOLO: clase y puntos x, y entre 0 y 1
    walls = [to_geometry(np.reshape(np.array(row[1:], dtype=float), (-1, 2)) * size) for row in rows if int(row[0]) == wall_id]
    walls = [wall for wall in walls if not wall.is_empty]
    if walls:
        measure_plan(results["Test de FloorPlanCAD"], image_path, size, walls)

## BLD-AR

In [ ]:
for json_path in sorted(BLDAR.glob("*.json")):
    data = json.loads(json_path.read_text())
    walls = [to_geometry(shape_points(s)) for s in data["shapes"] if s["label"] == "wall"]
    ignored = [to_geometry(shape_points(s)) for s in data["shapes"] if s["label"] not in MAPPED]
    walls = [wall for wall in walls if not wall.is_empty]
    if walls:
        measure_plan(results["BLD-AR"], BLDAR / data["imagePath"], (data["imageWidth"], data["imageHeight"]),
                     walls, [zone for zone in ignored if not zone.is_empty])

## AP50 de seg de pared de cada grupo

> Las paredes de los otros grupos, las que quedaron sin clasificar y las clases sin mapear de BLD-AR son zonas a ignorar: lo que el modelo predice sobre ellas no cuenta como acierto ni como error.

In [ ]:
def coco_annotation(geometry, image_id, iscrowd):
    x0, y0, x1, y1 = geometry.bounds
    return {
        "image_id": image_id, "category_id": 1, "iscrowd": iscrowd,
        "segmentation": [np.ravel(part.exterior.coords).tolist() for part in polygons(geometry)],
        "bbox": [x0, y0, x1 - x0, y1 - y0], "area": geometry.area,
    }


def ap50(result, style):
    """AP50 de seg de pared de un grupo, con COCOeval como en la evaluación en BLD-AR."""
    annotations = [coco_annotation(wall, image_id, iscrowd=int(wall_style != style))
                   for image_id, wall, wall_style, _ in result["walls"]]
    annotations += [coco_annotation(zone, image_id, iscrowd=1) for image_id, zone in result["ignored"]]
    for annotation_id, annotation in enumerate(annotations, start=1):
        annotation["id"] = annotation_id
    coco_gt = COCO()
    coco_gt.dataset = {"images": result["images"], "annotations": annotations, "categories": [{"id": 1, "name": "wall"}]}
    coco_gt.createIndex()
    # Copias: COCOeval reemplaza los polígonos de las predicciones por RLE
    evaluator = COCOeval(coco_gt, coco_gt.loadRes([dict(p) for p in result["predictions"]]), "segm")
    evaluator.evaluate()
    evaluator.accumulate()
    precision = evaluator.eval["precision"][0, :, 0, 0, -1]  # IoU 0.5, área "all", 100 detecciones
    return precision[precision > -1].mean()

## Resultado

> Pared encontrada: qué parte de cada pared etiquetada marca el modelo, en promedio. Solo en los planos con paredes etiquetadas.

In [ ]:
def found(result, style):
    return [value for _, _, wall_style, value in result["walls"] if wall_style == style]


text = ""
for name, result in results.items():
    text += f"**{name}**\n\n| Estilo | Paredes | Pared encontrada | AP50 de seg de pared |\n|---|---|---|---|\n"
    for style in STYLES:
        values = found(result, style)
        text += (f"| {style} | {len(values)} | {np.nanmean(values):.3f} | {ap50(result, style):.3f} |\n" if values
                 else f"| {style} | 0 | - | - |\n")
    text += f"| Sin clasificar (muy finas) | {len(found(result, None))} | - | - |\n\n"

fpc = results["Test de FloorPlanCAD"]
empty, with_lines = np.nanmean(found(fpc, "Vacía")), np.nanmean(found(fpc, "Con líneas adentro"))
text += (f"En el test de FloorPlanCAD, el modelo marca en promedio el {empty:.0%} de cada pared vacía (A) y el "
         f"{with_lines:.0%} de cada pared con líneas adentro (B): A / B = **{empty / with_lines:.0%}**.")
display(Markdown(f"**{WEIGHTS.parent.name}**\n\n" + text))